# Play with fine tuned models

In [1]:
import os
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig
# from datetime import datetime
# import matplotlib.pyplot as plt
# from dotenv import load_dotenv
import numpy as np
from torch.nn import functional as F

/Users/xing/Documents/personal/portfolio/BabyGPT/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load base model and adaptors

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


"baby-gpt-sft-tinystories-v1": r = 16, attn only, wandb 2026-04-24_22.15.40

"baby-gpt-sft-tinystories-v2": r = 32, attn only, wandb 2026-04-25_04.15.36

"baby-gpt-sft-tinystories-v3": r = 32, attn + mlp, wandb 2026-04-25_07.28.57

"baby-gpt-sft-tinystories-v4": r = 64, attn + mlp, wandb 2026-04-25_10.02.20

In [3]:
models = [
    "littleBallOfFur/baby-gpt-base",
    "littleBallOfFur/baby-gpt-sft-tinystories-v1",
    "littleBallOfFur/baby-gpt-sft-tinystories-v2",
    "littleBallOfFur/baby-gpt-sft-tinystories-v3",
    "littleBallOfFur/baby-gpt-sft-tinystories-v4",
]

In [4]:
tokenizer = AutoTokenizer.from_pretrained(models[0],trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [5]:
model_0 = AutoModelForCausalLM.from_pretrained(
    models[0], 
    trust_remote_code=True
)
model_1 = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(models[0],
    trust_remote_code=True), models[1]
)
model_2 = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(models[0],
    trust_remote_code=True), models[2]
)
model_3 = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(models[0],
    trust_remote_code=True), models[3]
)
model_4 = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(models[0],
    trust_remote_code=True), models[4]
)

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 19300.53it/s]


In [6]:
model_4

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): BabyGPTForCausalLM(
      (transformer): ModuleDict(
        (wte): Embedding(50304, 768)
        (wpe): Embedding(1024, 768)
        (h): ModuleList(
          (0-11): 12 x Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): CausalSelfAttention(
              (c_attn): lora.Linear(
                (base_layer): Linear(in_features=768, out_features=2304, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lo

## Test generation

In [7]:
def test_generate(model, prompts = [" "], max_new = 32, auto=True,temperature=1.0 ):
    torch.manual_seed(42)
    model.eval()
    # if generate in batch the padded ones are bad
    for i in range(len(prompts)):
        input_tokens = tokenizer.encode(prompts[i], return_tensors='pt')
        if auto:
            with torch.no_grad():
                output_tokens = model.generate(
                    input_tokens,
                    max_new_tokens=max_new,
                    do_sample=True,
                    temperature=temperature,
                    top_k=50,
                    top_p=0.95,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id,
                )
        else:
            for _ in range(max_new):
                # forward the model to get the logits
                with torch.no_grad():
                    logits = model(input_tokens).logits 
                    logits = logits[:, -1, :]/temperature # (B=1, vocab_size)
                    probs = F.softmax(logits, dim=-1)
                    topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
                    ix = torch.multinomial(topk_probs, 1) # (B, 1)
                    new_tok = torch.gather(topk_indices, -1, ix) # (B, 1)
                    input_tokens = torch.cat((input_tokens, new_tok), dim=1)

                    # v, _ = torch.topk(logits, min(50, logits.size(-1)))
                    # logits[logits < v[:, [-1]]] = -float('Inf')
                    # probs = F.softmax(logits, dim=-1)
                    # new_tok = torch.multinomial(probs, num_samples=1)
                    # input_tokens = torch.cat((input_tokens, new_tok), dim=1)
            output_tokens = input_tokens
        
        print(f'\n{prompts[i]}:\n{tokenizer.decode(output_tokens[0])}')



In [10]:
# torch.manual_seed(42)
prompts = [
    # "Once upon a time", 
    # "There was a little fat", 
    # "In a forest far far away",
    "A big fat squirrel and a small chipmunk"
]

test_generate(model_1,prompts,auto=True, max_new = 256 )

test_generate(model_2,prompts,auto=True, max_new = 256 )

test_generate(model_3,prompts,auto=True, max_new = 256 )

test_generate(model_4,prompts,auto=True, max_new = 256 )



A big fat squirrel and a small chipmunk:
A big fat squirrel and a small chipmunk were playing in the park. The squirrel said to the chipmunk, "Let's count up to 2, 4, 6, 8, 12, 15."
The chipmunk said, "That's right! A bit of ice is good for you." The chipmunk smiled and thought about it. It had a big head and a small tail. It was a great ice-breaker.
The chipmunk and the chipmunk got to play together and count up to two. They laughed and had a great time. The little squirrel and the big fat squirrel were happy. The squirrel said, "Now we have to try again."
The chipmunk said, "Okay. Let's go for a walk and count up again."
The little squirrel said, "Okay, here we go."<|endoftext|>

A big fat squirrel and a small chipmunk:
A big fat squirrel and a small chipmunk were playing in the park. Suddenly they heard a loud bang sound. They went closer and started to run to see what it was. 
The big fat squirrel said, "We are running away from that big pot!" 
The little chipmunk did not listen t

## Test story relay

In [11]:
def story_relay(models, prompt, each_turn_tokens=32, max_round=-1):
    for model in models:
        model.eval()
    tokens_so_far = tokenizer.encode(prompt, return_tensors='pt')
    print(f"\n{prompt}")
    if max_round > 0:
        for turn in range(max_round*len(models)):
            current_model = models[turn % len(models)] 
            with torch.no_grad():
                pre_length = tokens_so_far.shape[1]
                tokens_so_far = current_model.generate(
                    input_ids = tokens_so_far, 
                    max_new_tokens = each_turn_tokens,
                    do_sample = True,
                    top_k = 50,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id,
                )
            new_tokens = tokens_so_far[0,pre_length:]
            print(f"\nModel {turn % len(models) + 1} says:{tokenizer.decode(new_tokens)}")
    else:
        turn = 0
        while tokens_so_far.shape[1]<=1024-each_turn_tokens:
            current_model = models[turn % len(models)] 
            with torch.no_grad():
                pre_length = tokens_so_far.shape[1]
                tokens_so_far = current_model.generate(
                    input_ids = tokens_so_far, 
                    max_new_tokens = each_turn_tokens,
                    do_sample = True,
                    top_k = 50,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id,
                )
            new_tokens = tokens_so_far[0,pre_length:]
            print(f"\nModel {turn % len(models) + 1} says:{tokenizer.decode(new_tokens)}")
            turn += 1
            if tokenizer.eos_token_id in new_tokens.detach().cpu().tolist():
                break

    print("="*40)
    print(f"The full story is:\n{tokenizer.decode(tokens_so_far[0])}")
    print(f"Total number of tokens: {tokens_so_far.shape[1]}")


In [44]:
prompt = "A big fat squirrel flies"
story_relay(
    [model_1, model_2, model_3, model_4], 
    prompt, 
    each_turn_tokens=16, max_round=-1
)


A big fat squirrel flies

Model 1 says: into the sky. One by one, we see a rainbow coming through the sky

Model 2 says:. It is so colorful it reminds us that a strong storm is about to fall

Model 3 says:. We zoom and zoom and our eyes go dark for a while. Then,

Model 4 says: something wonderful happens. The sun's light shines and a small golden sun pops into

Model 1 says: the sky. It is so bright it gives us a great feeling of love and

Model 2 says: happiness.
At first, it was a gray, cloudless cloud. It

Model 3 says: was so nice and bright that everyone wanted to catch it. But then, something

Model 4 says: unexpected happened. The big fat squirrel saw a shiny rock in the sky. He

Model 1 says: noticed it was a magical spark. He got to touch the spark and then he

Model 2 says: got the magical spark. He laughed and got the spark. His enthusiasm was so

Model 3 says: high that he dared the clouds by running away. The sun was not very strong

Model 4 says: now and so he turned th